In [ ]:
from pathlib import Path
import copy
import sys
import json
from collections import defaultdict

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pypatchworkpp

from groundingdino.util.inference import load_model
from sam2.build_sam import build_sam2_video_predictor
from depth_anything_3.api import DepthAnything3
from depth_anything_3.utils.alignment import compute_sky_mask

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.common.nuscenes_utils import (
    get_scene_contents,
    get_sample_contents,
    get_sample_window_contents,
    get_sweep_images_in_sample,
    get_lidar_pointcloud_in_sample,
    get_sample_data_2d_bboxes,
    filter_category_group_bboxes,
    CATEGORY_MAPPING_TO_UNIAD
)
from src.common.visualize.detection import plot_2d_boxes_on_image
from src.common.visualize.segmentation import (
    plot_instance_masks_on_image,
    plot_instance_mask
)
from src.common.visualize.depth import plot_depth_map
from src.common.visualize.pointcloud import plot_pointcloud
from src.common.visualize.colors import TABLEAU10_NAMES
from src.common.schemas import Box2D
from src.common.frame_ops import create_sliding_windows
from src.common.image_processing.segmentation import mask_morphology
from src.common.geometry.crop_resize import resize_mask_nearest
from src.common.geometry.depth import (
    depth_map_to_point_cloud_per_instance,
    transform_cam_to_ego
)
from src.common.geometry.pointcloud import (
    transform_lidar_to_ego,
    transform_ego_to_global
)
from src.common.geometry.transform import (
    make_transform,
    invert_transform,
    quaternion_conjugate,
)

from src.grounding_dino.inference import predict_multi_labels
from src.sam2.inference import (
    init_frame_state,
    add_box_prompts,
    propagate_inference,
)
from src.sam2.utils import assign_continuous_tracking_ids
from src.depth_anything3.inference import get_metric_depth

# Resolve paths relative to this notebook directory
GROUNDINGDINO_CONFIG_PATH = ROOT / "GroundingDINO" / "groundingdino/config/GroundingDINO_SwinB_cfg.py"
GROUNDINGDINO_WEIGHT_PATH = ROOT / "GroundingDINO" / "weights/groundingdino_swinb_cogcoor.pth"
SAM2_CONFIG_PATH = "configs/sam2.1/sam2.1_hiera_l.yaml"
SAM2_CHECKPOINT_PATH = ROOT / "sam2" / "checkpoints/sam2.1_hiera_large.pt"
DA3_MODEL_NAME = "DA3METRIC-LARGE"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Display parameters
NUM_SHOWN = 3
SHOWN_CHANNEL = "CAM_FRONT"

# Build the GroundingDINO model
#groundingdino_model = load_model(str(GROUNDINGDINO_CONFIG_PATH), str(GROUNDINGDINO_WEIGHT_PATH), device=device)
# Build the SAM2 model and predictor
#sam2_predictor = build_sam2_video_predictor(str(SAM2_CONFIG_PATH), str(SAM2_CHECKPOINT_PATH), device=device)
# Build the Depth-Anything-3 model
#da3_model = DepthAnything3.from_pretrained(f"depth-anything/{DA3_MODEL_NAME}").to(device=device)
# Patchwork++
params = pypatchworkpp.Parameters()
PatchworkPLUSPLUS = pypatchworkpp.patchworkpp(params)
# Load nuScenes dataset
NUSCENES_ROOT = Path.cwd().parent / "data/nuscenes"
NUSCENES_VERSION = "v1.0-trainval"
CAMERA_CHANNELS = ["CAM_FRONT", "CAM_FRONT_RIGHT", "CAM_BACK_RIGHT", "CAM_BACK", "CAM_BACK_LEFT", "CAM_FRONT_LEFT"]
with open(NUSCENES_ROOT / NUSCENES_VERSION / "scene.json") as f:
    scenes = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample.json") as f:
    samples_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_data.json") as f:
    sample_data_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "ego_pose.json") as f:
    ego_poses_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "calibrated_sensor.json") as f:
    calibrated_sensors_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sensor.json") as f:
    sensors = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_annotation.json") as f:
    sample_annotations_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "instance.json") as f:
    instances_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "category.json") as f:
    categories = json.load(f)

# Create hash maps for token lookup
sensors = {sensor["token"]: sensor for sensor in sensors}
sensor_lookup = {sensor["channel"]: sensor["token"] for sensor in sensors.values()}

categories = {category["token"]: category for category in categories}
category_conversion = {k: v["category_name"] for k, v in CATEGORY_MAPPING_TO_UNIAD.items()}
category_names = list(dict.fromkeys(
    mapping["category_name"]
    for mapping in sorted(
        CATEGORY_MAPPING_TO_UNIAD.values(),
        key=lambda mapping: mapping["id"],
    )
))

# Get category groups
category_groups = set([v['category_group'] for v in CATEGORY_MAPPING_TO_UNIAD.values()])

print(f"original_category_names: {[category['name'] for category in categories.values()]}")
print(f"sensor_channels: {[sensor['channel'] for sensor in sensors.values()]}")
print(f"scene_names: {[scene['name'] for scene in scenes]}")

In [ ]:
# LiDAR Ground Removal by Patchwork++
# Select the scene
SCENE_NAME = "scene-0002"

NUM_LIDAR_SWEEPS = 5  # Number of LiDAR sweeps to use for point cloud projection


# Get the scene contents for the selected scene
scene = next(scene for scene in scenes if scene["name"] == SCENE_NAME)
scene_contents = get_scene_contents(scene["token"], samples_all, 
                                    sample_data_all, ego_poses_all, calibrated_sensors_all,
                                    get_non_key_frames=True,
                                    sample_annotations_all=sample_annotations_all,
                                    instances_all=instances_all,
                                    max_samples=10)
samples = scene_contents["samples"]
sample_data = scene_contents["sample_data"]
keyframe_sample_data = {k: v for k, v in sample_data.items() if v["is_key_frame"]}
ego_poses = scene_contents["ego_poses"]
calibrated_sensors = scene_contents["calibrated_sensors"]
sample_annotations = scene_contents["sample_annotations"]
instances = scene_contents["instances"]


pointclouds_per_instance = {}
# Frame loop
for sample_index in range(len(samples)):
    if sample_index < NUM_SHOWN:
        pointclouds_per_instance[sample_index] = {}

        # Read the LiDAR point cloud
        lidar_data = get_lidar_pointcloud_in_sample(
            sample_index=sample_index,
            samples_in_scene=samples,
            sample_data_in_scene=sample_data,
            ego_poses_in_scene=ego_poses,
            calibrated_sensors_in_scene=calibrated_sensors,
            lidar_sensor_token=sensor_lookup["LIDAR_TOP"],
            nuscenes_root=NUSCENES_ROOT,
            nsweeps=NUM_LIDAR_SWEEPS,
            stack_result=False,
        )
        # Estimate Ground
        ground = []
        nonground = []
        for sweep_lidar in lidar_data:
            pointcloud = np.hstack((sweep_lidar["points"], sweep_lidar["intensity"].reshape(-1, 1)))  # Combine points and intensity into a single array
            PatchworkPLUSPLUS.estimateGround(pointcloud)
            sweep_ground = PatchworkPLUSPLUS.getGround()
            sweep_nonground = PatchworkPLUSPLUS.getNonground()
            ground.append(sweep_ground)
            nonground.append(sweep_nonground)
        ground = np.vstack(ground)
        nonground = np.vstack(nonground)

        # Plot the pseudo-LiDAR point clouds
        ground_color = [1.0, 0.0, 0.0]
        nonground_color = [0.117647, 0.564706, 1.0]
        combilned_points = np.vstack((ground, nonground))
        colors = np.vstack((np.tile(ground_color, (ground.shape[0], 1)),
                            np.tile(nonground_color, (nonground.shape[0], 1))))
        ego_to_lidar = invert_transform(make_transform(lidar_data[-1]["lidar_quaternion"], lidar_data[-1]["lidar_translation"]))
        plot_fig = plot_pointcloud(combilned_points[:, :3],
                                colors,
                                axis_translation=ego_to_lidar[:3, 3].copy(),
                                axis_quaternion=quaternion_conjugate(np.asarray(lidar_data[-1]["lidar_quaternion"], dtype=np.float64)),
                                fig_width=960, fig_height=720,
                                point_size=1.0,
                                title=f"Sample {sample_index}, LiDAR + Pseudo-LiDAR point clouds"
                                )
        plot_fig.show()